In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Solve_atoms_with_alternate_locations)=
# Solve atoms with alternate locations

*Resolving alternate crystallographic atom locations in a molecular system.*

The function {func}`molsysmt.build.solve_atoms_with_alternate_location` resolves crystallographic alternate conformations by selecting a specific alternate location identifier (e.g. `'A'`, `'B'`, or the conformation with highest `'occupancy'`).

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.build.solve_atoms_with_alternate_location`.
:::


## Basic usage

Let's show how this function works using bundled PDB file `1BNF`:


In [2]:
import molsysmt as msm
from importlib.resources import files


In [3]:
pdb_filename = str(files('molsysmt.data.pdb').joinpath('1bnf.pdb'))
molsys = msm.convert(pdb_filename, to_form='molsysmt.MolSys', get_missing_bonds=False)


Let's inspect alternate location information for atom index `480` using {func}`molsysmt.basic.get`:


In [4]:
msm.get(molsys, element='atom', selection='atom_index==480', alternate_location=True)


[{480: {'location_id': array(['A', 'B'], dtype=object),
   'atom_id': ['481', '482'],
   'occupancy': array([0.5, 0.5]),
   'coordinates': <Quantity([[4.17   4.9768 4.186 ]
    [4.1655 4.9841 4.1835]], 'nanometer')>,
   'b_factor': <Quantity([0.2098 0.2083], 'nanometer ** 2')>}}]

Now we set all alternate locations to location ID `'B'` using {func}`molsysmt.build.solve_atoms_with_alternate_location`:


In [5]:
msm.build.solve_atoms_with_alternate_location(molsys, location_id='B')


In [6]:
msm.get(molsys, element='atom', selection=480, atom_id=True, coordinates=True)


[['482'], <Quantity([[[4.1655 4.9841 4.1835]]], 'nanometer')>]

## Resolving by maximum occupancy

Setting `location_id='occupancy'` selects the largest known crystallographic occupancy for every selected alternate atom. Occupancies are unit-free and are not renormalized. Unknown values are ignored; if every value is unknown, select a label explicitly. On a half-occupancy tie, A is preferred only when it shares the maximum (rtol=1e-5, atol=1e-8). Other ties keep source order. A=0.2 and B=0.5 therefore selects B.


In [7]:
molsys_occ = msm.convert(pdb_filename, to_form='molsysmt.MolSys', get_missing_bonds=False)
msm.build.solve_atoms_with_alternate_location(molsys_occ, location_id='occupancy')
msm.get(molsys_occ, element='atom', selection=480, atom_id=True, coordinates=True)


[['481'], <Quantity([[[4.17   4.9768 4.186 ]]], 'nanometer')>]

:::{warning} Alternate location metadata preservation
Native PDB file, PDB text, and `molsysmt.MolSys` forms retain alternate location metadata. When converting to third-party formats (such as MDAnalysis or OpenMM), alternate locations may be collapsed to the default conformer.
:::


## Keeping observations together

The chosen site's coordinates, B factor, occupancy and available atom-ID label become primary together. Missing optional measurements become unknown. The stable atom index does not change. When a request spans several structures, chosen IDs for each atom must agree because atom IDs are stable labels, not a per-structure series.


In [8]:
primary_occupancy = msm.get(molsys_occ, element='atom', selection=[480], occupancy=True)
print(primary_occupancy)
assert primary_occupancy.shape == (1, 1)


[[0.5]]


## Choosing a bounded scope

Use `selection` and explicit nonconsecutive `structure_indices` to limit the edit. Repeated structure positions are processed once. A vector of location labels follows the selected atom order, including a reordered numeric selection; repeated atoms must request the same label. Empty atom/structure selections do nothing. `skip_digestion=True` is only for already validated inputs and retains site preflight.

The complete selected request is checked before writing. A missing label in a later structure, inconsistent stable IDs, invalid occupancies, malformed quantities or missing writable domains leaves coordinates, labels, optional observations and attached analyses unchanged. Unexpected errors inside a mutable backend retain its own failure contract; this is not a transaction across arbitrary file formats.

A successful native coordinate edit marks only the affected structures unevaluated in attached interactions. Earlier analysis objects remain snapshots and unselected structures retain their evidence. Resolving alternatives neither recalculates interactions nor removes the original alternate records. Sparse planning does not copy the full coordinate series. Creating an absent primary occupancy/B-factor field uses its normal dense structure-by-atom storage, with unknown values outside the update.


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems between different forms with {func}`molsysmt.basic.convert`.
- {ref}`Get <Tutorial_Get>`: Query atom attributes and alternate location records with {func}`molsysmt.basic.get`.
- {ref}`Add missing heavy atoms <Tutorial_Add_missing_heavy_atoms>`: Add missing sidechain and backbone atoms with {func}`molsysmt.build.add_missing_heavy_atoms`.
:::
